# Kadanoff-Baym equations of motion

$$
\left[i \partial_z - h(z) \right]G(z,z^\prime) - \left[\Sigma * G\right](z,z^\prime) = \delta_C(z,z^\prime) \\
\left[-i \partial_z^\prime - h(z^\prime)\right]G(z,z^\prime) - \left[G * \Sigma\right](z,z^\prime)= \delta_C(z,z^\prime)
$$

In [ ]:
using KadanoffBaym
import NCA

include("../src/model.jl")
include("../src/wigner.jl")

using ElasticArrays
import Parameters: @unpack
import LinearAlgebra: diag
using Test

using Plots
default(fmt = :png)

## Initial correlations

### Equilibrium state

In [ ]:
siam = NCA.SingleImpurityTildeModel(
  V₀=sqrt(0.2),
  ε₀f=-0.5)

nca = NCA.FreqNCA(
  model=siam,
  β=5e2,
  mesh=NCA.LinearMesh(-3.0,3.0,2^13,false),
  eps=1e-9)

# Solve the problem in equilibrium
imΣ_t = NCA.solve!(nca; project=false)

# Calculate Green functions
GᶜA, GᶠA, GᵇA = begin
  Aᶜ = nca.data.Nc
  Gᶜ = NCA.kramerskronig(nca.mesh, π * Aᶜ, :real) + 1im * π * Aᶜ;
    
  Ginv = NCA.slave_particle_inv_gf(nca, imΣ_t)
  Gᵇ = 1. ./ Ginv[1]
  Gᶠ = 1. ./ Ginv[2]
    
  imGᵇ = view(reinterpret(Float64, Gᵇ),2:2:length(Gᵇ))
  imGᵇ[imGᵇ .< nca.eps] .= 0.0
    
  imGᶠ = view(reinterpret(Float64, Gᶠ),2:2:length(Gᶠ))
  imGᶠ[imGᶠ .< nca.eps] .= 0.0
    
  Gᶜ, Gᶠ, Gᵇ
end;

In [ ]:
# Create a new mesh that supports for complex convolutions
mesh = NCA.LinearMesh(nca.mesh.xmin, nca.mesh.xmax, nca.mesh.L, true);
bose = NCA.nb.(mesh, nca.β)
fermi = NCA.nf.(mesh, nca.β);
boltz = NCA.bf.(mesh.mesh, nca.β) .|> Float64 ; boltz[boltz .== Inf] .= 0.0;

∫dω(f) = trapz(mesh, f) / (2π)

# Assert GFs are normalized!
println(∫dω(GᶜA - conj(GᶜA)) |> imag)
println(∫dω(GᶠA - conj(GᶠA)) |> imag)
println(∫dω(GᵇA - conj(GᵇA)) |> imag)

plot(mesh.mesh, GᵇA |> imag, label=L"$A_b$", title="Spectral functions", xlab=L"$\omega$")
# plot!(mesh.mesh, GᶠA |> imag, label=L"$A_f$")
# plot!(mesh.mesh, GᶜA |> imag, label=L"$A_c$")

## Generating a gaussian pulse

Consider a gaussian pulse
$$\xi(t) = \left(\frac{\Omega^2}{2 \pi}\right)^{1/4}\exp{\left(-\frac{\Omega^2}{4}(t-t_c)^2\right)}$$

Consider a modulator cavity given by the triple $(S_M, L_M, H_M)$

for which $L_M = \lambda(t) a$ where $\lambda(t) \triangleq \frac{\xi(t)}{\sqrt{\int_t^\infty \textrm{d}s |\xi(s)|^2}}$

In [ ]:
import SpecialFunctions: erf, erfc

model = PhotonAssitedModel(
  #= SIAM parameters =#
  V₀=siam.V₀,
  ε₀f=siam.ε₀f,
  λ₀=siam.λ₀,
  σ=2.0,
  
  #= Light-field parameters =#
  Γ₀=0.2,  # light-matter coupling
  ω₀=1e-2, # photon-ground state energy
  δ₀=1e-1, # photon artificial width
  
  #= Photon-pulse parameters =#
  tc=40,   # peak arrival time
  Ω=1e-1,  # frequency bandwitdh of the pulse
  
  #= SLH photon formulation =#
  Γ=5.     # pump and loss strength
)

# Pulse and SLH shape
model.ξ = t -> (model.Ω^2 / (2π))^(1/4)* exp(-(t - model.tc)^2 * (model.Ω^2 / 4))
model.γ = t -> model.Γ * model.ξ(t)
model.λ = t -> model.Γ * model.ξ(t) / sqrt(0.5 * erfc((t - model.tc) * model.Ω / sqrt(2)))
model.hᵧ= t -> (model.ω₀ - 0.5im * (model.λ(t) + model.γ(t)))

In [ ]:
# TODO: using bose instead of boltz yields negative occupancy
data = begin
  Aᶜ = 2 * imag(GᶜA)
  Aᶠ = 2 * imag(GᶠA)
  Aᵇ = 2 * imag(GᵇA)
  n₀ = 0.0 # initial photon-field occupation  
  
  G₀ᶜL = +1.0im * ∫dω(fermi .* Aᶜ)
  G₀ᶜL = GreenFunction(G₀ᶜL * ones(1,1), Lesser)
  
  G₀ᶜG = -1.0im * (1 - ∫dω(fermi .* Aᶜ))
  G₀ᶜG = GreenFunction(G₀ᶜG * ones(1,1), Greater)
  
  G₀ᶠL = +1.0im * ∫dω(boltz .* Aᶠ)
  G₀ᶠL = GreenFunction(G₀ᶠL * ones(1,1), Lesser)

  G₀ᶠG = -1.0im * (1 - ∫dω(boltz .* Aᶠ))
  G₀ᶠG = GreenFunction(G₀ᶠG * ones(1,1), Greater)

  G₀ᵇL = -1.0im * ∫dω(boltz .* Aᵇ)
  G₀ᵇL = GreenFunction(G₀ᵇL * ones(1,1), Lesser)

  G₀ᵇG = -1.0im * (1 + ∫dω(boltz .* Aᵇ))
  G₀ᵇG = GreenFunction(G₀ᵇG * ones(1,1), Greater)
  
  G₀ᵞL = -1.0im * n₀
  G₀ᵞL = GreenFunction(G₀ᵞL * ones(1,1), Lesser)
  
  G₀ᵞG = -1.0im * (1 + n₀)
  G₀ᵞG = GreenFunction(G₀ᵞG * ones(1,1), Greater)
  
  Gᶜ˥m = GreenFunction(ElasticArray(hcat(GᶜA)), MixedLesser)
  Gᶜ˥p = GreenFunction(ElasticArray(hcat(conj(GᶜA))), MixedLesser)
  
  Gᶠ˥m = GreenFunction(ElasticArray(hcat(GᶠA)), MixedLesser)
  Gᶠ˥p = GreenFunction(ElasticArray(hcat(conj(GᶠA))), MixedLesser)
  
  Gᵇ˥m = GreenFunction(ElasticArray(hcat(GᵇA)), MixedLesser)
  Gᵇ˥p = GreenFunction(ElasticArray(hcat(conj(GᵇA))), MixedLesser)
  
  ProblemData(
    G₀ᶜL,G₀ᶜG,G₀ᶠL,G₀ᶠG,G₀ᵇL,G₀ᵇG,G₀ᵞL,G₀ᵞG,
    Gᶜ˥m,Gᶜ˥p,Gᶠ˥m,Gᶠ˥p,Gᵇ˥m,Gᵇ˥p,
    mesh,fermi,bose,boltz);
end;

In [ ]:
u0 = [
  [data.GᶜL,data.GᶜG,data.GᶠL,data.GᶠG,data.GᵇL,data.GᵇG,data.GᵞL,data.GᵞG],
  [data.Gᶜ˥m,data.Gᶜ˥p,data.Gᶠ˥m,data.Gᶠ˥p,data.Gᵇ˥m,data.Gᵇ˥p]
]

function f_vert(u, v, times, t1, t2)
  return f_vert_time(model, data, times, t1, t2) .+ f_vert_freq(model, data, times, t1, t2)
end

function f_diag(u, v, times, t1)
  println("t: $(times[t1])")
  return f_diag_time(model, data, times, t1) .+ f_diag_freq(model, data, times, t1)
end

function f_line(u, v, times, t1)
  return f_line_freq(model, data, times, t1)
end

In [ ]:
# ODE problem is defined by the rhs, initial condition and time span
tspan = (0.0,5)
u, times = kbsolve(f_vert, f_diag, u0, tspan[1], tspan[2]; f_line=f_line,
    dt=1e-4, max_dt=0.25,adaptive=true, atol=1e-9, rtol=1e-7)

In [ ]:
@testset "KMS boundary conditions" begin
  ≅(a,b) = isapprox(a,b;atol=1e-2)
  
  @unpack GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,Gᶜ˥m,Gᶜ˥p,Gᶠ˥m,Gᶠ˥p,Gᵇ˥m,Gᵇ˥p = data
  
  # G^R(t,t) = -i
  @test all(diag(GᶜG.data) - diag(GᶜL.data) .≅ -1.0im)
  @test all(diag(GᶠG.data) - diag(GᶠL.data) .≅ -1.0im)
  @test all(diag(GᵇG.data) - diag(GᵇL.data) .≅ -1.0im)
  @test all(diag(GᵞG.data) - diag(GᵞL.data) .≅ -1.0im)
  
  # C^<(t,0) = C˥(t,0+)
  @test GᶜL.data[:,1] ≅ [∫dω(fermi .* (Gᶜ˥m[t] - Gᶜ˥p[t])) for t in eachindex(times)]
  @test GᶠL.data[:,1] ≅ [∫dω(fermi .* (Gᶠ˥m[t] - Gᶠ˥p[t])) for t in eachindex(times)]
  @test GᵇL.data[:,1] ≅ [-∫dω(bose .* (Gᵇ˥m[t] - Gᵇ˥p[t])) for t in eachindex(times)]
end

In [ ]:
@unpack GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,Gᶜ˥m,Gᶜ˥p,Gᶠ˥m,Gᶠ˥p,Gᵇ˥m,Gᵇ˥p = data;

In [ ]:
plot(times, GᵞL.data |> diag .|> imag .|> -, label="Lesser")
plot!(times, GᵞG.data |> diag .|> imag .|> -, label="Greater")
plot!(twinx(), times, model.ξ.(times), color=:green, legend=:right, label="Pulse", line=:dot, xlabel=L"t")

In [ ]:
plot(imag(diag(GᵇL.data)))

In [ ]:
# heatmap(times, times, -imag(GᶜL.data))
# heatmap(times, times, -imag(GᶠL.data))
heatmap(times, times, -imag(GᵇL.data))
# heatmap(times, times, -imag(GᵞL.data))

In [ ]:
ρ = wigner_transform(GᵇG.data - GᵇL.data)#; Nf=4096*2);
wigner_plot!(real(ρ), times)#; ylims=(-10,10))

In [ ]:
ρ = wigner_transform(GᶠG.data - GᶠL.data)#; Nf=4096*2);
wigner_plot!(real(ρ), times)#; ylims=(-10,10))

In [ ]:
ρ = wigner_transform(GᵞG.data - GᵞL.data)#; Nf=4096*2);
wigner_plot!(real(ρ), times)#; ylims=(-3,3))